*Anual*\
https://dados.cvm.gov.br/dataset/cia_aberta-doc-dfp

💰 Lucro Líquido: dfp_cia_aberta_DRE_con_XXXX.csv (Conta 3.11)

🏢 Patrimônio Líquido: dfp_cia_aberta_BPP_con_XXXX.csv (Conta 2.03)

📈 Número de Ações: dfp_cia_aberta_composicao_capital_XXXX.csv

*Trimestre*\
https://dados.cvm.gov.br/dataset/cia_aberta-doc-itr

In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig
EnvConfig()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, trim, sqrt, when, lit, round as _round
from pyspark.sql.types import DecimalType

In [ ]:
# 1. Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("Cias Abertas: DFP") \
    .getOrCreate()

In [ ]:
CON = "asserts/dfp_cia_aberta_composicao_capital_2025.csv"
BPP_CON = "asserts/dfp_cia_aberta_BPP_con_2025.csv"
DRE = "asserts/dfp_cia_aberta_DRE_con_2025.csv"

In [ ]:
df_bpp = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(CON)
    
df_bpp_con = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(BPP_CON)
    
df_dre_con = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(DRE)

In [ ]:
df_bpp.show(truncate=False)

In [ ]:
df_bpp_con.show(truncate=False)

In [ ]:
df_dre_con.show(truncate=False)

In [ ]:
df_dre_con.describe()

In [ ]:
df_dre_con = (
    df_dre_con
    .filter(col("CD_CONTA") == "3.11")
    .withColumnRenamed("DENOM_CIA", "NOME CIA")
)

df_bpp_con = (
    df_bpp_con
    .filter(col("CD_CONTA") == "2.03")
    .withColumnRenamed("DENOM_CIA", "NOME CIA")
)

In [ ]:
lucro_acao = df_dre_con.join(df_bpp, on="CNPJ_CIA", how='inner').drop(df_bpp["VERSAO"])
valor_acao = df_bpp_con.join(df_bpp, on="CNPJ_CIA", how='inner').drop(df_bpp["VERSAO"])

In [ ]:
lucro_acao = (
    lucro_acao
    .withColumn("LPA - total", col("VL_CONTA")/col("QT_ACAO_TOTAL_CAP_INTEGR"))
    .withColumn(
            "LPA - pre", 
            when(col("QT_ACAO_PREF_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_PREF_CAP_INTEGR"))
        )
    .withColumn(
            "LPA - on",
            when(col("QT_ACAO_ORDIN_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_ORDIN_CAP_INTEGR"))
        )
)

valor_acao = (
    valor_acao
    .withColumn("VPA - total", col("VL_CONTA")/col("QT_ACAO_TOTAL_CAP_INTEGR"))
    .withColumn(
        "VPA - pre", 
        when(col("QT_ACAO_PREF_CAP_INTEGR") == 0, lit(0))
        .otherwise(col("VL_CONTA")/col("QT_ACAO_PREF_CAP_INTEGR"))
    )
    .withColumn(
            "VPA - on",
            when(col("QT_ACAO_ORDIN_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_ORDIN_CAP_INTEGR"))
        )
)


In [ ]:
valor_acao.show(truncate=False)

In [ ]:
vpa = (
    valor_acao
    .withColumnRenamed("ORDEM_EXERC", "ORDEM_EXERC_VPA")
    .withColumnRenamed("DT_FIM_EXERC", "DT_FIM_EXERC_VPA")
    .withColumnRenamed("VL_CONTA", "VL_CONTA_VPA")
    .select(
        col("CNPJ_CIA"),
        col("VL_CONTA_VPA"),
        col("VPA - on"),
        col("VPA - pre"),
        col("VPA - total"),
        col("ORDEM_EXERC_VPA"),
        col("DT_FIM_EXERC_VPA")
    )
)

In [ ]:
valor_acao.filter(col("CNPJ_CIA") == "17.281.106/0001-03").show(truncate=False)

In [ ]:
lucro_acao.show(truncate=False)

In [ ]:
lucro_acao.filter(col("CNPJ_CIA") == "17.281.106/0001-03").show(truncate=False)

In [ ]:
vpa.filter(col("CNPJ_CIA") == "17.281.106/0001-03").show(truncate=False)

In [ ]:
def metodo_graham(vpa, lpa):
    import math
    result = math.sqrt((22.5 * lpa * vpa))
    return result

In [ ]:
metodo_graham(21.165458260684336,3.4632731365695997)

In [ ]:
indicador = (
    lucro_acao.join(vpa, on=(lucro_acao["CNPJ_CIA"] == vpa["CNPJ_CIA"]) & 
    (lucro_acao["DT_FIM_EXERC"] == vpa["DT_FIM_EXERC_VPA"]), how="inner")
).drop(vpa["CNPJ_CIA"])

In [ ]:
indicador.describe()

In [ ]:
indicador = (
    indicador
    .withColumn(
        "Graham",
        sqrt(22.5 * (_round(col("VPA - total"),2)) * (_round(col("LPA - total"),2))).cast(DecimalType(38, 2))
    )
)

In [ ]:
indicador.filter(col("CNPJ_CIA") == "17.281.106/0001-03").show(truncate=False)

In [ ]:
final = (
    indicador
    .select(
            col("CNPJ_CIA"),
            col("DENOM_CIA"),
            col("CD_CVM"),
            col("ORDEM_EXERC"),
            col("MOEDA"),
            col("DT_INI_EXERC"),
            col("DT_FIM_EXERC"),
            col("VL_CONTA").alias("VL_LUCRO"),
            col("VL_CONTA_VPA").alias("VL_PATRIMONIO"),
            col("VPA - total"),
            col("LPA - total"),
            col("QT_ACAO_TOTAL_CAP_INTEGR"),
            col("Graham").alias("VL_GRAHAM_COTA")
        )
)

In [ ]:
final.filter(col("CNPJ_CIA") == "17.281.106/0001-03").show(truncate=False)